# IMDb DuckDB Pipeline: Many-to-Many Relationships

## What is a Many-to-Many (M2M) relationship?

A **many-to-many** relationship means:
- One movie can have **many writers**
- One writer can work on **many movies**

We **cannot** store this in a single table without duplication. So the proposed solution is **three tables**:

```
movies (tconst, primaryTitle, ...)   <──┐
                                        │  joined via tconst
writing_edges (tconst, writer_id)   ────┤
                                        │  joined via writer_id
writers (writer_id, name, ...)       <──┘  (optional, if you have name data)
```

The middle table (`writing_edges`) is called a **junction table** or **bridge table**. `writing.json` / `writing.csv` is already this table, it just needs to be loaded correctly.
For comparison, a one-to-many relationship would be something like movies → reviews: one movie has many reviews, but each review belongs to exactly one movie. You can just add a movie_id column to the reviews table. No junction table needed.

## Step 0: Environment & Paths

Update `BASE_PATH` to point to the folder where your `train-*.csv`, `writing.json`, and `directing.json` files live.

In [ ]:
import duckdb
import pandas as pd
import json
from pathlib import Path

BASE_PATH = Path("/Users/VanshitaS/Desktop/UVA - Period 4/Big Data/Big-Data/big_data_assignment/members/Vanshita/imdb")  # adjust if needed

csv_files      = sorted(BASE_PATH.glob("train-*.csv"))
writing_json   = BASE_PATH / "writing.json"
directing_json = BASE_PATH / "directing.json"

: 

## Step 1: Load all data into DuckDB

We create **three tables**:
- `imdb_train` — the movie metadata (one row per movie)
- `imdb_writing_edges` — writing junction table (one row per movie–writer pair)
- `imdb_directing_edges` — directing junction table (one row per movie–director pair)

> **Note:** `directing.json` uses a different format (columnar/pandas-style) than `writing.json`.
> We normalise it with pandas before loading into DuckDB.

In [ ]:
# Connect (in-memory — no file saved; use duckdb.connect('imdb.duckdb') to persist)
con = duckdb.connect()

# Load all train CSVs into one table 
# read_csv_auto with a glob pattern merges multiple files automatically
con.execute(f"""
    CREATE OR REPLACE TABLE imdb_train AS
    SELECT *
    FROM read_csv_auto('{BASE_PATH}/train-*.csv', HEADER=TRUE);
""")

# Load writing.json as the junction table 
# writing.json is an array of {{movie, writer}} objects 
# DuckDB reads this directly
con.execute(f"""
    CREATE OR REPLACE TABLE imdb_writing_edges AS
    SELECT
        movie   AS tconst,       -- rename to match imdb_train
        writer  AS writer_id
    FROM read_json_auto('{writing_json}');
""")

imdb_train rows  : 7,959
imdb_writing_edges rows: 22,428


In [5]:
# directing.json is in columnar/pandas format: {"movie": {"0": ..}, "director": {"0": ..}}
# This is DIFFERENT from writing.json (which was an array of objects).
# pandas.DataFrame() handles this format directly.
with open(directing_json) as f:
    raw_directing = json.load(f)

directing_df = pd.DataFrame(raw_directing).rename(
    columns={'movie': 'tconst', 'director': 'director_id'}
)

# Register the DataFrame as a DuckDB table
con.register('imdb_directing_edges', directing_df)

# ── Verify
n_dir_edges = con.execute("SELECT COUNT(*) FROM imdb_directing_edges").fetchone()[0]
n_edges     = con.execute("SELECT COUNT(*) FROM imdb_writing_edges").fetchone()[0]
n_movies    = con.execute("SELECT COUNT(*) FROM imdb_train").fetchone()[0]

print(f"imdb_train rows           : {n_movies:,}")
print(f"imdb_writing_edges rows   : {n_edges:,}")
print(f"imdb_directing_edges rows : {n_dir_edges:,}")

imdb_train rows           : 7,959
imdb_writing_edges rows   : 22,428
imdb_directing_edges rows : 11,162


## Step 2: Inspect the tables

In [6]:
print("imdb_train (first 5 rows)")
display(con.execute("SELECT * FROM imdb_train LIMIT 5").fetchdf())

imdb_train (first 5 rows)


,column0,tconst,primaryTitle,originalTitle,startYear,endYear,runtimeMinutes,numVotes,label
0,4,tt0010600,The Doll,Die Puppe,1919,\N,66,1898.0,True
1,7,tt0011841,Way Down East,Way Down East,1920,\N,145,5376.0,True
2,9,tt0012494,Déstiny,Der müde Tod,1921,\N,97,5842.0,True
3,25,tt0015163,The Navigator,The Navigator,1924,\N,59,9652.0,True
4,38,tt0016220,The Phantom of the Opera,The Phantom of the Opera,1925,\N,93,17887.0,True


In [7]:
print("imdb_writing_edges (first 10 rows)")
display(con.execute("SELECT * FROM imdb_writing_edges LIMIT 10").fetchdf())

imdb_writing_edges (first 10 rows)


,tconst,writer_id
0,tt0003740,nm0195339
1,tt0003740,nm0515385
2,tt0003740,nm0665163
3,tt0003740,nm0758215
4,tt0008663,nm0406585
5,tt0008663,nm0596410
6,tt0008663,nm0803705
7,tt0009369,nm0215874
8,tt0009369,nm0370271
9,tt0010307,nm0304098


In [8]:
print("imdb_directing_edges (first 10 rows)")
display(con.execute("SELECT * FROM imdb_directing_edges LIMIT 10").fetchdf())

imdb_directing_edges (first 10 rows)


,tconst,director_id
0,tt0003740,nm0665163
1,tt0008663,nm0803705
2,tt0009369,nm0428059
3,tt0009369,nm0949648
4,tt0010307,nm0304098
5,tt0010600,nm0523932
6,tt0011439,nm0629243
7,tt0011607,nm0003433
8,tt0011841,nm0000428
9,tt0012349,nm0000122


## Step 3: Validate the relationship

Check that every `tconst` in the writing edges actually exists in `imdb_train`.
Orphaned IDs mean the writing data references movies not in your training set.

In [12]:
orphans = con.execute("""
    SELECT COUNT(*) AS orphaned_edges
    FROM imdb_writing_edges w
    LEFT JOIN imdb_train t USING (tconst)
    WHERE t.tconst IS NULL
""").fetchone()[0]

total   = con.execute("SELECT COUNT(*) FROM imdb_writing_edges").fetchone()[0]
matched = total - orphans

print(f"Writing Total edges  : {total:,}")
print(f"Writing Matched      : {matched:,}  ({100*matched/total:.1f}%)")
print(f"Writing Orphaned     : {orphans:,}  (writing.json entries with no matching movie in train)")

Writing Total edges  : 22,428
Writing Matched      : 17,917  (79.9%)
Writing Orphaned     : 4,511  (writing.json entries with no matching movie in train)


In [13]:
orphans_dir = con.execute("""
    SELECT COUNT(*) AS orphaned_edges
    FROM imdb_directing_edges d
    LEFT JOIN imdb_train t USING (tconst)
    WHERE t.tconst IS NULL
""").fetchone()[0]

total_dir   = con.execute("SELECT COUNT(*) FROM imdb_directing_edges").fetchone()[0]
matched_dir = total_dir - orphans_dir

print(f"Directing Total edges  : {total_dir:,}")
print(f"Directing Matched      : {matched_dir:,}  ({100*matched_dir/total_dir:.1f}%)")
print(f"Directing Orphaned     : {orphans_dir:,}  (directing.json entries with no matching movie in train)")

Directing Total edges  : 11,162
Directing Matched      : 8,923  (79.9%)
Directing Orphaned     : 2,239  (directing.json entries with no matching movie in train)


## Step 4: Useful queries

### 4a. Join: get all writers per movie (aggregated into a list)

In [14]:
df_joined = con.execute("""
    SELECT
        t.tconst,
        t.primaryTitle,
        t.startYear,
        t.label,
        COUNT(w.writer_id)          AS writer_count,
        string_agg(w.writer_id, ', ')       AS writer_ids
    FROM imdb_train t
    LEFT JOIN imdb_writing_edges w USING (tconst)
    GROUP BY t.tconst, t.primaryTitle, t.startYear, t.label
    ORDER BY writer_count DESC
    LIMIT 10
""").fetchdf()

display(df_joined)

,tconst,primaryTitle,startYear,label,writer_count,writer_ids
0,tt0401711,"Paris, je t'aime",2006,True,29,"nm0138894, nm0687913, nm0074488, nm0149446, nm..."
1,tt0120855,Tarzan,\N,True,25,"nm0614742, nm0879318, nm0925276, nm0027459, nm..."
2,tt1333125,Mớvié 43,2013,False,20,"nm2415313, nm2326580, nm0088546, nm3913565, nm..."
3,tt1935896,Thé ÁBCs ớf Déáth,2012,False,20,"nm0863807, nm1443023, nm0305563, nm1104993, nm..."
4,tt0032138,The Wizard of Oz,1939,True,19,"nm0486538, nm0753249, nm0941138, nm0000875, nm..."
5,tt0119282,Hercules,1997,True,18,"nm0166256, nm0615780, nm0568490, nm0789624, nm..."
6,tt4116284,The Lego Batman Movie,2017,True,14,"nm0334381, nm0571344, nm1273099, nm2972864, nm..."
7,tt10695464,The Source of Shadows,2020,False,14,"nm3423183, nm1769926, nm7121645, nm4287898, nm..."
8,tt5022680,All Hallows' Eve 2,2015,False,14,"nm2156502, nm2629535, nm2795237, nm3965884, nm..."
9,tt3896198,Gúárdiáns ớf thé Gáláxy Vớl. 2,2017,True,13,"nm0348181, nm3966115, nm5037683, nm1921680, nm..."


### 4b. Most prolific writers (by number of movies)

In [15]:
df_prolific = con.execute("""
    SELECT
        writer_id,
        COUNT(DISTINCT tconst) AS movie_count
    FROM imdb_writing_edges
    GROUP BY writer_id
    ORDER BY movie_count DESC
    LIMIT 10
""").fetchdf()

display(df_prolific)

,writer_id,movie_count
0,\N,297
1,nm0372942,25
2,nm0000636,25
3,nm0440604,18
4,nm0080327,16
5,nm0000697,15
6,nm0006249,15
7,nm0000095,15
8,nm0000041,15
9,nm0346096,15


### 4c. Movies with the most writers (potential collaborations)

In [13]:
df_collab = con.execute("""
    SELECT
        t.primaryTitle,
        t.startYear,
        COUNT(w.writer_id) AS writer_count
    FROM imdb_train t
    JOIN imdb_writing_edges w USING (tconst)
    GROUP BY t.tconst, t.primaryTitle, t.startYear
    ORDER BY writer_count DESC
    LIMIT 10
""").fetchdf()

display(df_collab)

,primaryTitle,startYear,writer_count
0,"Paris, je t'aime",2006,29
1,Tarzan,\N,25
2,Thé ÁBCs ớf Déáth,2012,20
3,Mớvié 43,2013,20
4,The Wizard of Oz,1939,19
5,Hercules,1997,18
6,The Source of Shadows,2020,14
7,The Lego Batman Movie,2017,14
8,All Hallows' Eve 2,2015,14
9,Gúárdiáns ớf thé Gáláxy Vớl. 2,2017,13


### 4d. Join: get all directors per movie

In [16]:
df_dir_joined = con.execute("""
    SELECT
        t.tconst,
        t.primaryTitle,
        t.startYear,
        t.label,
        COUNT(d.director_id)              AS director_count,
        string_agg(d.director_id, ', ')   AS director_ids
    FROM imdb_train t
    LEFT JOIN imdb_directing_edges d USING (tconst)
    GROUP BY t.tconst, t.primaryTitle, t.startYear, t.label
    ORDER BY director_count DESC
    LIMIT 10
""").fetchdf()

display(df_dir_joined)

,tconst,primaryTitle,startYear,label,director_count,director_ids
0,tt1687247,Life in a Day,2011,True,35,"nm0531817, nm3946993, nm7470817, nm4309887, nm..."
1,tt1935896,Thé ÁBCs ớf Déáth,2012,False,27,"nm3094978, nm3464186, nm0871188, nm0774143, nm..."
2,tt0401711,"Paris, je t'aime",2006,True,21,"nm0001814, nm0878756, nm0858680, nm0840485, nm..."
3,tt1333125,Mớvié 43,2013,False,13,"nm0348181, nm0711840, nm2086105, nm0268380, nm..."
4,tt2275743,"Berlin, I Love You",2019,False,13,"nm0874795, nm1127653, nm0001709, nm0750857, nm..."
5,tt10695464,The Source of Shadows,2020,False,12,"nm1634724, nm3738224, nm4464117, nm9798074, nm..."
6,tt12607910,Black Is King,\N,False,12,"nm11870049, nm6003581, nm8887027, nm6290395, n..."
7,tt5022680,All Hallows' Eve 2,2015,False,11,"nm0636171, nm2795237, nm0002911, nm4539541, nm..."
8,tt0092580,Aria,1987,False,10,"nm0854697, nm0836430, nm0000419, nm0734466, nm..."
9,tt1539997,Kerala Cafe,\N,True,10,"nm3488838, nm0430485, nm2057169, nm2776729, nm..."


### 4e. Most prolific directors (by number of movies)

In [17]:
df_dir_prolific = con.execute("""
    SELECT
        director_id,
        COUNT(DISTINCT tconst) AS movie_count
    FROM imdb_directing_edges
    WHERE director_id != '\\N'
    GROUP BY director_id
    ORDER BY movie_count DESC
    LIMIT 10
""").fetchdf()

display(df_dir_prolific)

,director_id,movie_count
0,nm0000033,19
1,nm0000406,17
2,nm0006249,15
3,nm0089502,14
4,nm0654868,13
5,nm0001238,13
6,nm0000095,13
7,nm0002030,13
8,nm0943758,13
9,nm0000485,13


## Step 5: Feature engineering for ML

If you're using this for a classification task (predicting `label`), here's how to turn the many-to-many relationship into useful features per movie.

## Summary

| Table | Rows | Description |
|---|---|---|
| `imdb_train` | one per movie | Movie metadata + label |
| `imdb_writing_edges` | one per movie×writer pair | Writing junction table |
| `imdb_directing_edges` | one per movie×director pair | Directing junction table |

**The key pattern:** Join via `tconst`, use `GROUP BY + aggregates` to collapse the many writers/directors back to one row per movie.

**Format difference to remember:**
- `writing.json` → array of objects → `read_json_auto()` works directly
- `directing.json` → columnar (pandas-style) → normalise with `pd.DataFrame()` first, then `con.register()`
